# 01 · Adquisición y congelamiento de datos — Grupo 17
**Proyecto Integrador MIAR0545 · Paso 1 del Sprint 2**

Objetivo del paso: descargar las tres fuentes, congelarlas con hash SHA-256 y levantar el inventario.
Este paso **no** hace EDA, **no** armoniza etiquetas y **no** limpia nada; eso corresponde al Paso 3.

**Criterios de cierre (evidencia exigida):**
1. `manifest.json` con todos los archivos y su SHA-256.
2. Una segunda ejecución en modo verificación: 0 discrepancias de hash.
3. PhiUSIIL: 235.795 filas y etiqueta `1` = legítimo (134.850) confirmadas.
4. Inventario del dataset de Kaggle: archivos, filas, columnas, nulos y valores de etiqueta por archivo.
5. El bloque `RESUMEN` pegado en la conversación.

Funciona igual en **Colab** y en **Kaggle**: la primera celda detecta el entorno.

In [2]:
# Celda 1: Entorno, rutas y semillas
import os, sys, json, hashlib, random, platform, shutil, re, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 17
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

def detectar_entorno() -> str:
    """colab | kaggle | local. No depende de rutas, solo de marcas del runtime."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"

ENV = detectar_entorno()

# Nombre del snapshot congelado. Si cambian los datos, se sube la versión (v2, v3...).
SNAPSHOT = "g17-phishing-raw-v1"

if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/PI_G17")          # Drive = copia maestra
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))

RAW  = BASE / "data" / "raw" / SNAPSHOT
META = BASE / "data" / "meta"
for p in (RAW, META):
    p.mkdir(parents=True, exist_ok=True)

# En Kaggle, si el snapshot ya fue publicado como dataset privado, se lee de ahí (solo lectura).
KAGGLE_SNAPSHOT_INPUT = Path(f"/kaggle/input/{SNAPSHOT}")
MODO = "verificar_snapshot_kaggle" if (ENV == "kaggle" and KAGGLE_SNAPSHOT_INPUT.exists()) else "adquirir"

print(f"Entorno: {ENV} | Modo: {MODO}")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"RAW  = {RAW}\nMETA = {META}")

Mounted at /content/drive
Entorno: colab | Modo: adquirir
Python 3.13.15 | pandas 2.2.3 | numpy 2.1.3
RAW  = /content/drive/MyDrive/PI_G17/data/raw/g17-phishing-raw-v1
META = /content/drive/MyDrive/PI_G17/data/meta


In [3]:
# Celda 2: Dependencias (solo gratuitas)
# En Kaggle: Settings -> Internet ON (requiere cuenta verificada por teléfono).
if MODO == "adquirir":
    %pip install -q kagglehub ucimlrepo requests

In [4]:
# Celda 3: Utilidades de hash y copia
def sha256_archivo(path: Path, bloque: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bloque), b""):
            h.update(chunk)
    return h.hexdigest()

def inventario_hashes(raiz: Path) -> dict:
    """{ruta_relativa: {sha256, bytes}} de todos los archivos bajo raiz, en orden estable."""
    out = {}
    for p in sorted(raiz.rglob("*")):
        if p.is_file():
            out[p.relative_to(raiz).as_posix()] = {"sha256": sha256_archivo(p), "bytes": p.stat().st_size}
    return out

def ahora_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()

### Fuente 1 · Kaggle Phishing Email Dataset
Si `kagglehub` devuelve 401/403 en Colab: crear en el panel **Secrets** (icono de llave) `KAGGLE_USERNAME` y `KAGGLE_KEY`
(se obtienen en kaggle.com → Settings → API → Create New Token) y habilitarles acceso al notebook.

In [5]:
# Celda 4: Kaggle Phishing Email Dataset
KAGGLE_REF = "naserabdullahalam/phishing-email-dataset"
DEST_KAGGLE = RAW / "kaggle_phishing_email"
fuentes = {}

if MODO == "adquirir":
    if ENV == "colab":
        try:
            from google.colab import userdata
            os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
            os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
        except Exception as e:
            print("Aviso: sin secrets de Kaggle; se intenta sin credenciales ->", type(e).__name__)
    import kagglehub
    src = Path(kagglehub.dataset_download(KAGGLE_REF))
    DEST_KAGGLE.mkdir(parents=True, exist_ok=True)
    for f in sorted(src.rglob("*")):
        if f.is_file():
            destino = DEST_KAGGLE / f.relative_to(src)
            destino.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, destino)
    fuentes["kaggle_phishing_email"] = {
        "origen": f"https://www.kaggle.com/datasets/{KAGGLE_REF}",
        "ruta_cache_kagglehub": str(src),   # contiene el número de versión descargada
        "licencia": "Ver ficha de Kaggle; citar además Champa, Rabbi y Zibran (2024) como fuente académica",
        "descargado_utc": ahora_utc(),
    }
    print("Archivos:", [p.name for p in sorted(DEST_KAGGLE.rglob('*')) if p.is_file()])
    print("Cache (anotar la versión):", src)

Aviso: sin secrets de Kaggle; se intenta sin credenciales -> SecretNotFoundError


100%|██████████| 77.1M/77.1M [00:00<00:00, 105MB/s] 

Extracting files...


Archivos: ['CEAS_08.csv', 'Enron.csv', 'Ling.csv', 'Nazario.csv', 'Nigerian_Fraud.csv', 'SpamAssasin.csv', 'phishing_email.csv']
Cache (anotar la versión): /root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1


In [6]:
# Celda 5: PhiUSIIL (UCI id 967)
# Si la API de UCI falla (caída intermitente conocida): descargar el ZIP a mano desde la ficha
# del dataset en UCI, subirlo a RAW/phiusiil/ y saltar esta celda.
DEST_PHI = RAW / "phiusiil"
if MODO == "adquirir":
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=967)
    df_phi = getattr(ds.data, "original", None)
    if df_phi is None:
        df_phi = pd.concat([ds.data.features, ds.data.targets], axis=1)
    DEST_PHI.mkdir(parents=True, exist_ok=True)
    df_phi.to_csv(DEST_PHI / "phiusiil_original.csv", index=False)
    fuentes["phiusiil"] = {
        "origen": "UCI Machine Learning Repository, id 967 (vía ucimlrepo)",
        "licencia": "CC BY 4.0",
        "cita": "Prasad & Chandra (2024), Computers & Security",
        "nota": "El hash corresponde al CSV serializado desde ucimlrepo, no al ZIP original.",
        "descargado_utc": ahora_utc(),
    }
    print(df_phi.shape)

(235795, 56)


### Fuente 3 · SpamAssassin crudo (opcional, no bloqueante)
El dataset de Kaggle ya trae `SpamAssasin.csv`. El crudo solo aporta encabezados completos para validar el extractor.
Si esta celda falla, se desactiva con `DESCARGAR_SPAMASSASSIN = False` y el paso sigue cerrable.

In [7]:
# Celda 6: SpamAssassin Public Corpus (tarballs crudos)
DESCARGAR_SPAMASSASSIN = True
SA_URL = "https://spamassassin.apache.org/old/publiccorpus/"
DEST_SA = RAW / "spamassassin_raw"
if MODO == "adquirir" and DESCARGAR_SPAMASSASSIN:
    import requests
    UA = {"User-Agent": "UEES-MIAR0545-Grupo17-research/1.0"}
    html = requests.get(SA_URL, headers=UA, timeout=60).text
    # Se lee el listado real del directorio en vez de fijar nombres de archivo a mano.
    tarballs = sorted(set(re.findall(r'href="([^"/]+\.tar\.bz2)"', html)))
    DEST_SA.mkdir(parents=True, exist_ok=True)
    for name in tarballs:
        dst = DEST_SA / name
        if not dst.exists():
            r = requests.get(SA_URL + name, headers=UA, timeout=300); r.raise_for_status()
            dst.write_bytes(r.content)
    for extra in ("readme.html",):   # condiciones de uso del corpus
        r = requests.get(SA_URL + extra, headers=UA, timeout=60)
        if r.ok: (DEST_SA / extra).write_bytes(r.content)
    fuentes["spamassassin_raw"] = {"origen": SA_URL, "archivos": tarballs,
                                   "licencia": "Ver readme.html del directorio",
                                   "descargado_utc": ahora_utc()}
    print(f"{len(tarballs)} tarballs:", tarballs)

9 tarballs: ['20021010_easy_ham.tar.bz2', '20021010_hard_ham.tar.bz2', '20021010_spam.tar.bz2', '20030228_easy_ham.tar.bz2', '20030228_easy_ham_2.tar.bz2', '20030228_hard_ham.tar.bz2', '20030228_spam.tar.bz2', '20030228_spam_2.tar.bz2', '20050311_spam_2.tar.bz2']


In [12]:
# Celda 7: Congelar o verificar (manifest.json)
# Primera ejecución: crea el manifest. Ejecuciones siguientes: SOLO verifican, nunca sobrescriben.
MANIFEST = META / f"manifest_{SNAPSHOT}.json"
raiz = KAGGLE_SNAPSHOT_INPUT if MODO == "verificar_snapshot_kaggle" else RAW

def verificar(manifest: dict, raiz: Path) -> dict:
    actual = inventario_hashes(raiz)
    esperado = manifest["archivos"]
    faltan    = sorted(set(esperado) - set(actual))
    sobran    = sorted(set(actual) - set(esperado) - {"manifest.json"})
    distintos = sorted(k for k in set(esperado) & set(actual) if esperado[k]["sha256"] != actual[k]["sha256"])
    return {"faltan": faltan, "sobran": sobran, "hash_distinto": distintos,
            "ok": not (faltan or sobran or distintos), "n_archivos": len(actual)}

if MODO == "verificar_snapshot_kaggle":
    MANIFEST = KAGGLE_SNAPSHOT_INPUT / "manifest.json"

if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text())
    resultado_verificacion = verificar(manifest, raiz)
    print("MODO VERIFICACIÓN ->", resultado_verificacion)
else:
    manifest = {"snapshot": SNAPSHOT, "creado_utc": ahora_utc(), "entorno": ENV,
                "fuentes": fuentes, "archivos": inventario_hashes(raiz)}
    MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    shutil.copy2(MANIFEST, RAW / "manifest.json")   # viaja dentro del snapshot
    resultado_verificacion = None
    print(f"Manifest creado: {len(manifest['archivos'])} archivos -> {MANIFEST}")
    print("Vuelve a ejecutar ESTA celda para obtener la verificación (criterio 2).")

MODO VERIFICACIÓN -> {'faltan': [], 'sobran': [], 'hash_distinto': [], 'ok': True, 'n_archivos': 19}


In [9]:
# Celda 8: Inventario (sin limpiar ni transformar nada)
def leer_csv_robusto(path: Path) -> pd.DataFrame:
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer {path}")

def perfil_csv(path: Path) -> dict:
    df = leer_csv_robusto(path)
    etiqueta = next((c for c in df.columns if c.lower() in ("label", "labels", "class", "target")), None)
    return {
        "archivo": path.name, "filas": int(len(df)), "columnas": list(map(str, df.columns)),
        "nulos_pct": {c: round(float(v), 2) for c, v in (df.isna().mean() * 100).items() if v > 0},
        "col_etiqueta": etiqueta,
        "valores_etiqueta": ({str(k): int(v) for k, v in df[etiqueta].value_counts(dropna=False).items()}
                             if etiqueta else None),
    }

inventario = [perfil_csv(p) for p in sorted(raiz.rglob("*.csv"))]
pd.DataFrame([{k: v for k, v in i.items() if k in ("archivo", "filas", "col_etiqueta", "valores_etiqueta")}
              for i in inventario])

,archivo,filas,col_etiqueta,valores_etiqueta
0,CEAS_08.csv,39154,label,"{'1': 21842, '0': 17312}"
1,Enron.csv,29767,label,"{'0': 15791, '1': 13976}"
2,Ling.csv,2859,label,"{'0': 2401, '1': 458}"
3,Nazario.csv,1565,label,{'1': 1565}
4,Nigerian_Fraud.csv,3332,label,{'1': 3332}
5,SpamAssasin.csv,5809,label,"{'0': 4091, '1': 1718}"
6,phishing_email.csv,82486,label,"{'1': 42891, '0': 39595}"
7,phiusiil_original.csv,235795,label,"{'1': 134850, '0': 100945}"


In [10]:
# Celda 9: Chequeo explícito de PhiUSIIL (criterio 3)
ESPERADO_PHI = {"filas": 235795, "legitimo_1": 134850, "phishing_0": 100945}
phi = next((i for i in inventario if i["archivo"].startswith("phiusiil")), None)
if phi is None:
    chequeo_phi = {"ok": False, "motivo": "PhiUSIIL no encontrado"}
else:
    v = phi["valores_etiqueta"] or {}
    chequeo_phi = {
        "filas": phi["filas"], "valores": v,
        "ok": phi["filas"] == ESPERADO_PHI["filas"]
              and v.get("1") == ESPERADO_PHI["legitimo_1"]
              and v.get("0") == ESPERADO_PHI["phishing_0"],
    }
print(chequeo_phi)

{'filas': 235795, 'valores': {'1': 134850, '0': 100945}, 'ok': True}


In [11]:
# Celda 10: RESUMEN (copiar todo lo que imprime)
resumen = {
    "entorno": ENV, "modo": MODO, "snapshot": SNAPSHOT,
    "n_archivos_manifest": len(manifest["archivos"]),
    "verificacion": resultado_verificacion,
    "chequeo_phiusiil": chequeo_phi,
    "inventario": inventario,
    "fuentes": {k: {kk: vv for kk, vv in v.items() if kk != "archivos"} for k, v in manifest.get("fuentes", {}).items()},
}
print("===== RESUMEN =====")
print(json.dumps(resumen, indent=1, ensure_ascii=False, default=str))

===== RESUMEN =====
{
 "entorno": "colab",
 "modo": "adquirir",
 "snapshot": "g17-phishing-raw-v1",
 "n_archivos_manifest": 18,
 "verificacion": null,
 "chequeo_phiusiil": {
  "filas": 235795,
  "valores": {
   "1": 134850,
   "0": 100945
  },
  "ok": true
 },
 "inventario": [
  {
   "archivo": "CEAS_08.csv",
   "filas": 39154,
   "columnas": [
    "sender",
    "receiver",
    "date",
    "subject",
    "body",
    "label",
    "urls"
   ],
   "nulos_pct": {
    "receiver": 1.18,
    "subject": 0.07
   },
   "col_etiqueta": "label",
   "valores_etiqueta": {
    "1": 21842,
    "0": 17312
   }
  },
  {
   "archivo": "Enron.csv",
   "filas": 29767,
   "columnas": [
    "subject",
    "body",
    "label"
   ],
   "nulos_pct": {
    "subject": 0.67
   },
   "col_etiqueta": "label",
   "valores_etiqueta": {
    "0": 15791,
    "1": 13976
   }
  },
  {
   "archivo": "Ling.csv",
   "filas": 2859,
   "columnas": [
    "subject",
    "body",
    "label"
   ],
   "nulos_pct": {
    "subject": 2

### Después de cerrar el paso: publicar el snapshot en Kaggle
1. En kaggle.com → *Datasets* → *New Dataset*, privado, con el nombre exacto `g17-phishing-raw-v1`.
2. Subir el contenido de `MyDrive/PI_G17/data/raw/g17-phishing-raw-v1/` (incluye `manifest.json`).
3. En cualquier notebook de Kaggle: *Add Input* → ese dataset. La Celda 1 lo detecta y la Celda 7 verifica los hashes.
Si los hashes coinciden en ambos entornos, queda demostrado que Colab y Kaggle entrenan sobre los mismos bytes.